# 🤖 Module 3 · Class 6 — Olist Capstone (AI Prompt Edition)

**The new way to work:** you do **not** write Python from memory, and you do **not** copy blanks.
For each step you **write a clear prompt**, an **AI** (ChatGPT / Claude / Gemini) writes the code,
and **you paste it, run it, and check the result**.

**Why?** Writing good prompts is a real job skill. By the end of this lab you will know how to make
an AI build a full data pipeline for you — and how to *check* that it did it right.

**Your final product:** the file `olist_clean.parquet` (Module 4 will train on it).

**Level:** A2 English. Solo or in pairs.

## 📦 The dataset you are working with

**Olist** = a real online shop in Brazil. The data is **8 tables (CSV files)**:

| Table (variable) | What is inside | Key columns |
| --- | --- | --- |
| `orders` | each order + its dates | order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date |
| `customers` | who bought + location | customer_id, customer_unique_id, customer_state, customer_zip_code_prefix |
| `items` | products in each order | order_id, order_item_id, seller_id, price, freight_value |
| `payments` | how it was paid | order_id, payment_type, payment_installments, payment_value |
| `reviews` | rating + comment | order_id, review_score, review_comment_message, review_creation_date |
| `sellers` | the seller + location | seller_id, seller_state, seller_zip_code_prefix |
| `geo` | zip code → map point | geolocation_zip_code_prefix, geolocation_lat, geolocation_lng |
| `products` | product details (barely used) | product_id, ... |

**The job:** join the useful pieces into ONE table (one row per order) and build the target
**`is_late`** (1 = the order arrived after the estimated date, 0 = on time).

> You will give the AI these table + column names in your prompts. That is why they are listed here.

## 🧠 The most important skill: the PROMPT RECIPE

A good prompt has **4 parts**. If you miss one, the AI guesses wrong.

| Part | What you write |
| --- | --- |
| **1. Context** | the dataset + which tables/columns you have |
| **2. Goal** | what you need this step to produce |
| **3. What you already have** | the variables that exist from earlier steps |
| **4. Ask** | "write pandas code, comment each line, and tell me the expected result" |

### 📋 Copy this template — reuse it for EVERY stage
```
I'm working in Python with pandas on the Olist Brazilian e-commerce dataset.
These DataFrames are already loaded: orders, customers, items, payments, reviews, sellers, geo.

GOAL (this step): <write what you need to produce>
DETAILS: <the exact columns / any filter / how to combine>
WHAT I ALREADY HAVE: <variables made in earlier steps>

Please write the pandas code, add a short comment on each line, and then tell me
what shape or number I should expect so I can check it.
```

### ❌ Bad prompt → *"write code to clean olist data"* (AI has no idea what you mean)
### ✅ Good prompt → fills in all 4 parts with the real table and column names.

**Every stage below tells you exactly what to put in GOAL and DETAILS. You write the prompt,
paste the AI's code into the empty cell, run it, and check the result.**

## Stage 1 — Load the 8 tables

📋 **Your task:** load all 8 CSV files into DataFrames named `orders, customers, items, payments,
reviews, products, sellers, geo`.

📦 **Give the AI these facts in your prompt:**
- The files are online. The base address is:
  `https://huggingface.co/datasets/aviahYadler/Olist_Ecommerce_Dataset/resolve/main`
- The file names are like `olist_orders_dataset.csv`, `olist_customers_dataset.csv`, … (one per table).
- You want to read each one with `pd.read_csv(BASE + '/' + filename)`.
- After loading, print each table's `.shape`.

✍️ **Write your prompt** (double-click to type), then paste the AI's code in the cell below and run it.

In [1]:
# 👇 Paste the code the AI gave you here, then run it (Shift+Enter).
import pandas as pd, numpy as np
import pandas as pd

# Define the base URL for the dataset files
BASE = "https://huggingface.co/datasets/aviahYadler/Olist_Ecommerce_Dataset/resolve/main"

# Load each dataset into a DataFrame
orders = pd.read_csv(f"{BASE}/olist_orders_dataset.csv")      # Load orders
customers = pd.read_csv(f"{BASE}/olist_customers_dataset.csv")  # Load customers
items = pd.read_csv(f"{BASE}/olist_order_items_dataset.csv")    # Load items
payments = pd.read_csv(f"{BASE}/olist_order_payments_dataset.csv") # Load payments
reviews = pd.read_csv(f"{BASE}/olist_order_reviews_dataset.csv")   # Load reviews
products = pd.read_csv(f"{BASE}/olist_products_dataset.csv")    # Load products
sellers = pd.read_csv(f"{BASE}/olist_sellers_dataset.csv")      # Load sellers
geo = pd.read_csv(f"{BASE}/olist_geolocation_dataset.csv")      # Load geolocation

# Print the shape of each DataFrame to verify the load
print(f"Orders: {orders.shape}")
print(f"Customers: {customers.shape}")
print(f"Items: {items.shape}")
print(f"Payments: {payments.shape}")
print(f"Reviews: {reviews.shape}")
print(f"Products: {products.shape}")
print(f"Sellers: {sellers.shape}")
print(f"Geo: {geo.shape}")


Orders: (99441, 8)
Customers: (99441, 5)
Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geo: (1000163, 5)


✅ **Check your result:** you should see `orders: (99441, 8)`, `geo: (1000163, 5)`, and 6 more.
If the AI's code errors, **paste the error back to it** and ask it to fix it (that is part 4 of the recipe).

## Stage 2 — Clean the `orders` table

📋 **Your task:** make `orders` ready: real dates, only useful orders, no missing delivery dates.

📦 **Put this in your prompt (GOAL + DETAILS):**
- Turn these columns into real dates: `order_purchase_timestamp`, `order_approved_at`,
  `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`
  (use `pd.to_datetime(..., errors='coerce')`).
- Keep only rows where `order_status` is one of `delivered, invoiced, shipped, approved` → call it `orders_model`.
- Drop rows in `orders_model` that have no `order_delivered_customer_date` or no `order_estimated_delivery_date`.
- Print how many rows remain.

✍️ Write the prompt, paste the code below, run it.

In [2]:
# 👇 Paste the AI's code for Stage 2 here.
# List of columns to convert to datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Convert the specified columns to datetime objects
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Filter for relevant order statuses
orders_model = orders[
    orders['order_status'].isin(['delivered', 'invoiced', 'shipped', 'approved'])
].copy()

# Drop rows where critical delivery date information is missing
orders_model = orders_model.dropna(
    subset=['order_delivered_customer_date', 'order_estimated_delivery_date']
)

# Verify the result
print(f"Final shape of orders_model: {orders_model.shape}")


Final shape of orders_model: (96470, 8)


✅ **Check:** about **96,470** rows remain (roughly 1,431 dropped). If your number is very different,
tell the AI what you got and ask why.

## Stage 3 — One row per order (aggregate items + payments)

📋 **Your task:** items and payments have many rows per order. Squeeze them to one row per order.

📦 **Put this in your prompt:**
- From `items`: per `order_id`, make `num_items` (count of `order_item_id`), `total_price` (sum of `price`),
  `total_freight` (sum of `freight_value`). Call it `order_totals`.
- From `payments`: keep the **biggest** payment per `order_id` (its `payment_type`, `payment_installments`).
  Call it `primary_pay`.
- From `items`: the **first** seller per `order_id` (`order_id`, `seller_id`). Call it `first_seller`.

✍️ Write the prompt, paste the code, run it.

In [3]:
# 👇 Paste the AI's code for Stage 3 here.
# Aggregate items to get total items, price, and freight per order
order_totals = items.groupby('order_id').agg(
    num_items=('order_item_id', 'count'),       # Count the number of items
    total_price=('price', 'sum'),               # Sum the price
    total_freight=('freight_value', 'sum')      # Sum the freight value
).reset_index()

# Keep the largest payment per order_id to identify the primary payment method
# Sort by order_id and payment_value, then drop duplicates to keep the highest value
primary_pay = payments.sort_values(['order_id', 'payment_value'], ascending=False).drop_duplicates('order_id')
# Select only relevant columns
primary_pay = primary_pay[['order_id', 'payment_type', 'payment_installments']]

# Get the first seller per order_id
first_seller = items.groupby('order_id').first().reset_index()[['order_id', 'seller_id']]

# Print shapes for verification
print(f"order_totals shape: {order_totals.shape}")
print(f"primary_pay shape: {primary_pay.shape}")
print(f"first_seller shape: {first_seller.shape}")


order_totals shape: (98666, 4)
primary_pay shape: (99440, 3)
first_seller shape: (98666, 2)


✅ **Check:** `order_totals` ≈ (98666, 4), `primary_pay` ≈ (99440, 3), `first_seller` ≈ (98666, 2).

## Stage 4 — Distance from seller to customer

📋 **Your task:** build a tool that gives the distance in km between two map points, and a lookup
that turns a zip code into one map point.

📦 **Put this in your prompt:**
- Write a `haversine(lat1, lon1, lat2, lon2)` function that returns the distance in **km** (Earth radius 6371).
- From `geo`: one **average** `lat`/`lon` per `geolocation_zip_code_prefix`. Call it `geo_lookup`.

✍️ Write the prompt, paste the code, run it.

> 💡 The haversine math is tricky — this is a perfect thing to let the AI write. Just **check** the result looks right.

In [11]:
# 👇 Paste the AI's code for Stage 4 here.
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    # Earth's radius in kilometers
    R = 6371
    # Convert decimal degrees to radians
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    # Apply Haversine formula
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))
# shu joyida AI natijada adashdi. Birinchi urunishda natija geo_lookup shape: (19015, 2) chiqdi. keyin qayta to'g'rilandi.
# Group by zip code, calculate the mean, and reset index to move the zip code into a column
geo_lookup = geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

# Print the shape to verify
print(f"geo_lookup shape: {geo_lookup.shape}")

geo_lookup shape: (19015, 3)


✅ **Check:** `geo_lookup` ≈ (19015, 3) — one row per zip code.

## Stage 5 — Glue everything + build the features (the big one)

📋 **Your task:** start from `orders_model`, merge everything, compute distance, dates, and `is_late`.

📦 **Put this in your prompt (this one is long — give ALL of it):**
- Start with `df = orders_model`. Merge on, using `how='left'`:
  `customers` (on `customer_id`), `order_totals` (`order_id`), `primary_pay` (`order_id`),
  `first_seller` (`order_id`), `sellers` (`seller_id`).
- Add customer lat/lon and seller lat/lon by merging `geo_lookup` on the two zip-code columns.
- `distance_km` = `haversine(cust_lat, cust_lon, sell_lat, sell_lon)`.
- `delivery_days` = delivered − purchase (in days); `estimate_days` = estimated − purchase (in days).
- `is_late` = 1 if delivered **after** estimated, else 0.
- From the purchase date make: `purchase_year, purchase_month, purchase_dayofweek, purchase_hour`,
  `is_weekend` (1 if dayofweek ≥ 5), and `log_freight` = `np.log1p(total_freight)`.

✍️ Write the prompt, paste the code, run it.

> 💡 If the prompt feels too big, split it: ask for the merges first, run it, then ask for the features.

In [14]:
# 👇 Paste the AI's code for Stage 5 here.
import numpy as np

# 1. Merge core tables into one master DataFrame
df = orders_model.merge(customers, on='customer_id', how='left')
df = df.merge(order_totals, on='order_id', how='left')
df = df.merge(primary_pay, on='order_id', how='left')
df = df.merge(first_seller, on='order_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')

# 2. Merge geo coordinates for customers and sellers
df = df.merge(geo_lookup, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df = df.rename(columns={'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lon'}).drop(columns='geolocation_zip_code_prefix')
df = df.merge(geo_lookup, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df = df.rename(columns={'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lon'}).drop(columns='geolocation_zip_code_prefix')

# 3. Calculate distance using the haversine function
df['distance_km'] = haversine(df['cust_lat'], df['cust_lon'], df['sell_lat'], df['sell_lon'])

# 4. Calculate delivery metrics (days)
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400
df['estimate_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400

# 5. Create 'is_late' flag (1 if delivered after estimated date)
df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)

# 6. Extract temporal features
df['purchase_year'] = df['order_purchase_timestamp'].dt.year
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour
df['is_weekend'] = (df['purchase_dayofweek'] >= 5).astype(int)

# 7. Apply log transformation to freight
df['log_freight'] = np.log1p(df['total_freight'])

# Print final shape
print(f"Final shape of df: {df.shape}")

Final shape of df: (96470, 35)


✅ **Check:** `df` has ~96,470 rows; `df['is_late'].mean()` ≈ **0.08**; `df['distance_km']` averages ~600 km.

## Stage 6 — Add reviews + save the final file

📋 **Your task:** attach the review, keep the 21 needed columns, save `olist_clean.parquet`.

📦 **Put this in your prompt:**
- From `reviews`: one review per `order_id` (the first), keep `review_score`, `review_comment_message`.
  Merge it onto `df` (`how='left'`).
- Keep ONLY these 21 columns (give the AI this exact list):
  `order_id, customer_unique_id, customer_state, seller_state, purchase_year, purchase_month,`
  `purchase_dayofweek, purchase_hour, is_weekend, num_items, total_price, total_freight, log_freight,`
  `payment_type, payment_installments, distance_km, delivery_days, estimate_days, is_late,`
  `review_score, review_comment_message`
- Save as `olist_clean.parquet` with `index=False`. Print the shape and the `is_late` rate.

✍️ Write the prompt, paste the code, run it.

In [15]:
# 👇 Paste the AI's code for Stage 6 here.
# Aggregate reviews to keep one record per order_id (using the first entry)
reviews_agg = reviews.groupby('order_id')[['review_score', 'review_comment_message']].first().reset_index()

# Merge the review data onto your existing master DataFrame 'df'
df = df.merge(reviews_agg, on='order_id', how='left')

# Define the exact list of 21 columns to keep
columns_to_keep = [
    'order_id', 'customer_unique_id', 'customer_state', 'seller_state',
    'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour',
    'is_weekend', 'num_items', 'total_price', 'total_freight', 'log_freight',
    'payment_type', 'payment_installments', 'distance_km', 'delivery_days',
    'estimate_days', 'is_late', 'review_score', 'review_comment_message'
]

# Filter the DataFrame to keep only those columns
df_clean = df[columns_to_keep]

# Save the final cleaned DataFrame to a Parquet file
df_clean.to_parquet('olist_clean.parquet', index=False)

# Print final shape and the 'is_late' rate
print(f"Final shape: {df_clean.shape}")
print(f"Is_late rate: {df_clean['is_late'].mean():.2%}")

Final shape: (96470, 21)
Is_late rate: 8.11%


✅ **Check:** `olist_clean.parquet` has shape **(96470, 21)**, `is_late` rate ≈ **0.08**, reviews ≈ 99%.
**This is your final file. Do not lose it.**

## Stage 7 — Findings (write this yourself, NOT with AI)

Double-click and fill in (graded):

1. **Total rows in `olist_clean.parquet`:** 96470
2. **`is_late` rate:** 8.11 (about 0.08)
3. **Three things you noticed while doing this:** ______ / ______ / ______
4. **One prompt that did NOT work the first time — what did you change to fix it?** ______

> Question 4 is the real lesson: getting a prompt to work on the second or third try **is** the skill.

## 📤 Submission

Put **your notebook AND `olist_clean.parquet`** in:
```
module-3/class_6/submissions/<YourTeamName>/
```

### 🎓 Grading (100 points)
| What we check | Points |
| --- | --- |
| Final file: 21 correct columns | 25 |
| Your prompts are clear (the 4 parts) — paste them in a text cell | 20 |
| Notebook runs top-to-bottom with no error | 20 |
| You **checked** each result against the ✅ targets | 15 |
| Findings filled in (incl. the fixed-prompt story) | 10 |
| `is_late` rate is 5–10% | 10 |

### Remember the recipe: **Context → Goal → What I have → Ask.** Give all 4, every time.

🏁 *You just made an AI build a real data pipeline for you — and you checked its work. That is the job.*